In [1]:
from unstructured.partition.pdf import partition_pdf

c:\Users\tchouarr\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
from pathlib import Path
import re

from unstructured.partition.pdf import partition_pdf

from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

from langchain_core.documents import Document

from sentence_transformers import SentenceTransformer

from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

In [30]:
from dotenv import load_dotenv

In [31]:
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [2]:
from pathlib import Path
from unstructured.partition.pdf import partition_pdf

# Chemin vers le dossier contenant les PDF
data_dir = Path("../data")



In [3]:
# Récupère tous les fichiers PDF du dossier
pdf_files = data_dir.glob("*.pdf")

# Dictionnaire pour stocker les résultats
all_elements = {}

for pdf_file in pdf_files:
    print(f"Traitement : {pdf_file.name}")

    elements = partition_pdf(
        filename=str(pdf_file),
        strategy="fast",
        extract_images_in_pdf=False,
        languages=["fra"]
    )

    # Stockage des éléments par nom de fichier
    all_elements[pdf_file.name] = elements

print("Tous les PDF ont été traités.")

Traitement : 123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf
Traitement : doc_.pdf
Traitement : Guide paramétrage restrictions de visibilité agenda (1).pdf
Traitement : Guide Utilisateur HM Agenda v1.24.11.pdf
Traitement : Paramétrage Etablissement FILIERIS.pdf
Traitement : [GU] Calcul des besoins V12.pdf
Traitement : [GU] Cotation à partir de l'agenda - 1.2307.pdf
Tous les PDF ont été traités.


In [4]:
len(all_elements)

7

In [5]:
all_elements

{'123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf': [<unstructured.documents.elements.Text at 0x25434727710>,
 'doc_.pdf': [<unstructured.documents.elements.Title at 0x254399f9750>,
 'Guide paramétrage restrictions de visibilité agenda (1).pdf': [<unstructured.documents.elements.Footer at 0x25439aebd50>,
 'Guide Utilisateur HM Agenda v1.24.11.pdf': [<unstructured.documents.elements.Footer at 0x2543a4dca50>,
  ...],
 'Paramétrage Etablissement FILIERIS.pdf': [<unstructured.documents.elements.Header at 0x2543a0dff90>,
 '[GU] Calcul des besoins V12.pdf': [<unstructured.documents.elements.Title at 0x2543c788d10>,
 "[GU] Cotation à partir de l'agenda - 1.2307.pdf": [<unstructured.documents.elements.Title at 0x25439f82d90>,
  <unstructured.documents.elements.Footer at 0x2543a14fdd0>]}

In [6]:
elements[:5]

## checking metadata

In [7]:
for pdf_name, elements in all_elements.items():

    print(f"\n===== {pdf_name} =====\n")

    for el in elements[:50]:
        print(type(el))
        print(el.text)
        print(el.metadata)
        print("=" * 50)


===== 123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf =====

<class 'unstructured.documents.elements.Text'>
Table des matières 1 INTRODUCTION ..........................................................................................................................................2
<class 'unstructured.documents.elements.Text'>
2 EXERCICE N°1 – Planning de ressource ......................................................................................................2
<class 'unstructured.documents.elements.Text'>
2 EXERCICE N°2 – Prise de RDV .....................................................................................................................3
<class 'unstructured.documents.elements.Footer'>
1
<class 'unstructured.documents.elements.Footer'>
Cahier d’exercices – AG002 – Agenda (en moyen et long séjour)
<class 'unstructured.documents.elements.NarrativeText'>
Ce cahier d’exercices a pour objectif d’aider à la validation des compétences C0188 et C0202 de la formation

In [8]:
markdown_text = ""

for pdf_name, elements in all_elements.items():

    # Titre du document
    markdown_text += f"\n\n# 📄 {pdf_name}\n\n"

    current_page = None

    for el in elements:

        # Numéro de page
        page = getattr(el.metadata, "page_number", None)

        # Affiche le changement de page
        if page != current_page:
            current_page = page
            markdown_text += f"\n\n---\n\n## Page {page}\n\n"

        # Conversion selon le type d’élément
        if el.category == "Title":
            markdown_text += f"\n### {el.text}\n"

        elif el.category == "ListItem":
            markdown_text += f"- {el.text}\n"

        elif el.category == "NarrativeText":
            markdown_text += f"\n{el.text}\n"

        elif el.category == "Table":
            markdown_text += f"\n```text\n{el.text}\n```\n"

        else:
            markdown_text += f"\n{el.text}\n"

In [9]:
markdown_text

'\n\n# 📄 123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf\n\n\n\n---\n\n## Page 2\n\n\nTable des matières 1 INTRODUCTION ..........................................................................................................................................2\n\n2 EXERCICE N°1 – Planning de ressource ......................................................................................................2\n\n2 EXERCICE N°2 – Prise de RDV .....................................................................................................................3\n\n1\n\nCahier d’exercices – AG002 – Agenda (en moyen et long séjour)\n\n\n---\n\n## Page 3\n\n\nCe cahier d’exercices a pour objectif d’aider à la validation des compétences C0188 et C0202 de la formation AG002- Agenda (en moyen et long séjour).\n\n### Référence de l’exercice AG002-EXO1 Validation de la compétence C0202\n\nPremier pas\n\na) Accéder aux agendas de type « Planning de ressource » et choisissez-en un.\n\nb) Sur l’agenda

In [10]:
elements

In [11]:
from langchain_core.documents import Document

docs = []

for pdf_name, elements in all_elements.items():

    current_header = "Sans titre"

    for el in elements:

        # Mise à jour du header
        if el.category == "Title":
            current_header = el.text.strip()
            continue

        # Sécurise le texte
        text = getattr(el, "text", "")

        if not text:
            continue

        text = text.strip()

        if not text:
            continue

        # Page
        page = getattr(el.metadata, "page_number", None)

        # Création du document LangChain
        doc = Document(
            page_content=text,
            metadata={
                "header": current_header,
                "page": page,
                "category": el.category,
                "source": pdf_name
            }
        )

        docs.append(doc)

In [12]:
docs

[Document(metadata={'header': 'Sans titre', 'page': 2, 'category': 'UncategorizedText', 'source': '123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf'}, page_content='Table des matières 1 INTRODUCTION ..........................................................................................................................................2'),
 Document(metadata={'header': 'Sans titre', 'page': 2, 'category': 'UncategorizedText', 'source': '123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf'}, page_content='2 EXERCICE N°1 – Planning de ressource ......................................................................................................2'),
 Document(metadata={'header': 'Sans titre', 'page': 2, 'category': 'UncategorizedText', 'source': '123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf'}, page_content='2 EXERCICE N°2 – Prise de RDV .....................................................................................................................3'),
 Document(metadata

In [13]:
len(docs)

2966

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

In [15]:
final_docs = splitter.split_documents(docs)

In [16]:
final_docs

[Document(metadata={'header': 'Sans titre', 'page': 2, 'category': 'UncategorizedText', 'source': '123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf'}, page_content='Table des matières 1 INTRODUCTION ..........................................................................................................................................2'),
 Document(metadata={'header': 'Sans titre', 'page': 2, 'category': 'UncategorizedText', 'source': '123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf'}, page_content='2 EXERCICE N°1 – Planning de ressource ......................................................................................................2'),
 Document(metadata={'header': 'Sans titre', 'page': 2, 'category': 'UncategorizedText', 'source': '123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf'}, page_content='2 EXERCICE N°2 – Prise de RDV .....................................................................................................................3'),
 Document(metadata

In [17]:
len(final_docs)

3137

In [18]:
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [19]:
from dotenv import load_dotenv

In [20]:
# Embedding wrapper for LangChain
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 20048.08it/s]


In [21]:
dimension = 1024

In [22]:
persist_directory="all_data2"

In [23]:
chroma_vdb=Chroma.from_documents(documents=final_docs,
                      embedding=embedding_model,
                      persist_directory=persist_directory,
                      )

In [24]:
chroma_vdb.persist()

C:\Users\tchouarr\AppData\Local\Temp\ipykernel_1944\3808622788.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma_vdb.persist()


### appel de la base

In [25]:
vdb=Chroma(persist_directory=persist_directory,
        embedding_function=embedding_model)

C:\Users\tchouarr\AppData\Local\Temp\ipykernel_1944\35172397.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vdb=Chroma(persist_directory=persist_directory,


In [26]:
retriever=vdb.as_retriever(search_kwargs={"k": 30})

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=""
)

In [29]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [33]:
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant spécialisé dans les documents administratifs français.

Réponds uniquement à partir du contexte fourni.

Contexte:
{context}

Question:
{question}
Si possible de donné le numero de la page et le document qui contient l'information
Si l'information n'est pas présente dans le contexte,
réponds:
"Information non trouvée dans les documents."
""")

In [34]:
chain = prompt | llm | StrOutputParser()

In [35]:
question = "comment calculer le besoin d'un site ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le calcul du besoin d'un site peut être effectué en prenant en compte les données de plusieurs sites (ceux appartenant au groupe de site). Pour ce faire, il est possible de déclarer le groupe de site dans les données de tous les sites appartenant au groupe. Les données de consommations, du disponible, des encours fournisseurs, des encours services sont cumulées selon tous les sites du groupe.

Le calcul du besoin net est effectué comme suit :

- Besoin brut = Consommation de la période explorée / nb jours de la période explorée

Il est également possible de calculer le besoin selon un seuil de réappro manuel avec besoin = seuil réappro.

Ces informations sont disponibles dans le contexte fourni, mais sans référence à une page ou un document spécifique. 

Il est important de noter que le calcul du besoin peut varier en fonction du type de délai (1, 2 ou SRM) et des paramètres définis dans les fiches fournisseur et article.


In [36]:

question = "comment calculer le besoin ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le calcul du besoin net est effectué en fonction de plusieurs paramètres, notamment le type de délai et les données de consommation. 

La formule de calcul du besoin brut est la suivante : 
→ Besoin brut = Consommation de la période explorée / nb jours de la période explorée

Les paramètres utilisés pour le calcul du besoin net sont les suivants :
- XNBJRCBNS1 – Nb jr conso sans délai (CBN) : nombre de jours par défaut pour le calcul du besoin net pour les articles sans délai (Type de délai dans la fiche article = blanc).
- XNBJRCBNS2 – Nb jr conso délai 1 (CBN) : nombre de jours par défaut pour le calcul du besoin net pour les articles ayant un type de délai = 1 ou D1 (Type de délai dans la fiche article = 1 ou D1).
- XNBJRCBNS3 – Nb jr conso délai 2 (CBN) : nombre de jours par défaut pour le calcul du besoin net pour les articles ayant un type de délai = 2 ou D2 (Type de délai dans la fiche article = 2 ou D2).

Malheureusement, le numéro de la page et le document qui contiennent l'in

In [38]:
def format_docs(docs):

    formatted = []

    for doc in docs:

        page = doc.metadata.get("page", "Unknown")
        header = doc.metadata.get("header", "Sans titre")
        source = doc.metadata.get("source", "Unknown")

        chunk = f"""
Source: {source}
Page: {page}
Section: {header}

Contenu:
{doc.page_content}
"""

        formatted.append(chunk)

    return "\n\n".join(formatted)

In [39]:
context = format_docs(retrieved_docs)

NameError: name 'retrieved_docs' is not defined

In [40]:
question = "est-il possible d’avoir une visualisation multi-agents ?(1 thérapeute/colonne lorsqu’on sélectionne un jour)"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Oui, il est possible d'avoir une visualisation multi-agents. Selon le document "Guide Utilisateur HM Agenda v1.24.11.pdf" à la page 131, il est possible de définir les ressources, c'est-à-dire les Professionnels/Salles/Groupes de patients à afficher en en-tête de colonne, pour créer les plannings de rendez-vous et/ou de vacations avec personnalisation de l'affichage.

Cela signifie que vous pouvez avoir une colonne par thérapeute ou par ressource, ce qui permet une visualisation multi-agents.

Référence : Guide Utilisateur HM Agenda v1.24.11.pdf, page 131.


In [41]:
question = "Comment permettre d’effectuer un changement de thérapeute pour un patient sur plusieurs jours, actuellement limité à une modification jour par jour sans suppression et recréation de la série de rendez-vous ?"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

La fonctionnalité pour changer le professionnel de santé de la séance est disponible, mais elle est limitée à une modification jour par jour sans suppression et recréation de la série de rendez-vous. Cependant, il est possible de dupliquer une à une les séances d’un seul patient via l’action « Dup » dans la section "Guide Utilisateur HM Agenda" (Page 115 du document Guide Utilisateur HM Agenda v1.24.11.pdf).

Cela permet de modifier les séances individuellement, mais il n'y a pas de fonctionnalité pour changer de thérapeute pour un patient sur plusieurs jours en une seule opération.

Réponse:
Page 115 du document Guide Utilisateur HM Agenda v1.24.11.pdf.


In [44]:
question = "Quelle est la différence exacte entre : UF mentionnées et  Toutes UFs sauf celles mentionnées  ?"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

La différence exacte entre "UF mentionnées" et "Toutes UFs sauf celles mentionnées" est la suivante :

- "UF mentionnées" : L'utilisateur aura accès qu'aux UFs mentionnées en dessous dans le tableau des UF accessibles. (Page 5, Guide paramétrage restrictions de visibilité agenda (1).pdf)
- "Toutes UFs sauf celles mentionnées" : L'utilisateur aura accès à toutes les UFs sauf celles qui seront mentionnées en dessous dans le tableau des UF accessibles. (Page 5, Guide paramétrage restrictions de visibilité agenda (1).pdf)

En résumé, la première option restreint l'accès aux UFs spécifiquement mentionnées, tandis que la deuxième option permet l'accès à toutes les UFs sauf celles qui sont explicitement exclues.


In [45]:
question = "Quelle est la règle d’affichage spécifique des vacations lorsqu’une restriction UF est activée ?"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

La règle d'affichage spécifique des vacations lorsqu'une restriction UF est activée est la suivante : 
On n'affiche que les vacations dont l'UFH paramétrée est accessible par l'utilisateur.

Cette information se trouve dans le document "Guide paramétrage restrictions de visibilité agenda (1).pdf" à la page 6, section "Règles d’affichage".


In [46]:



question = "Quels comportements du système pourraient entraîner des erreurs d’interprétation chez un modèle RAG ?"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Les comportements du système qui pourraient entraîner des erreurs d'interprétation chez un modèle RAG incluent :

- La cotation de l'activité pour les actes du Recueil ACTES&DIAG, qui peut être faite dès la planification du rendez-vous et non pas au moment de l'accueil du rendez-vous, ce qui peut être problématique si le rendez-vous n'a finalement pas eu lieu (Page 68, Guide Utilisateur HM Agenda v1.24.11.pdf).
- La prise de rendez-vous manuelle sur l'agenda patient, qui se faisait par défaut sur le séjour sur lequel on est positionné dans la synthèse générale avant la version 1.15.09, ce qui pouvait entraîner des erreurs d'interprétation (Page 153, Guide Utilisateur HM Agenda v1.24.11.pdf).
- La création de séjour via la prise de rendez-vous, qui peut entraîner une annonce ou une admission, selon le paramétrage de RDV_STATUT_SEJOUR (Page 152, Guide Utilisateur HM Agenda v1.24.11.pdf).

Cependant, il est important de noter que le contexte fourni ne mentionne pas explicitement les erreu

In [47]:


question = "Existe-t-il une hiérarchie entre restriction établissement, UF et PS ?"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Il n'y a pas d'information explicite sur une hiérarchie entre restriction établissement, UF et PS dans les documents fournis. Cependant, on peut noter que les restrictions de visibilité peuvent être paramétrées par UF et par PS, ainsi que par établissement, mais sans indication claire d'une hiérarchie entre ces niveaux de restriction.

On peut citer la page 11 du document "Guide paramétrage restrictions de visibilité agenda (1).pdf" qui mentionne les restrictions de visibilité côté RPL basées sur le paramétrage des UF et PS accessibles par l’utilisateur, mais sans évoquer une hiérarchie.

Réponse:
Information non trouvée dans les documents.


In [48]:


question = "Les restrictions sont-elles appliquées avant ou après la recherche RPL ?"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Information non trouvée dans les documents.


In [49]:



question = "Résume uniquement les restrictions liées aux UF sans parler des PS."

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Les restrictions liées aux UF sont les suivantes :

* L'utilisateur aura accès à toutes les UFs sauf celles qui seront mentionnées (Page 5, Guide paramétrage restrictions de visibilité agenda (1).pdf)
* L'utilisateur aura accès qu'aux UFs mentionnées en dessous dans le tableau des UF (Page 5, Guide paramétrage restrictions de visibilité agenda (1).pdf)
* Il est possible de restreindre l'accès à des agendas auxquels l'utilisateur est abonné en utilisant les UF accessibles de l'utilisateur (Page 219, Guide Utilisateur HM Agenda v1.24.11.pdf)
* Vous avez la possibilité de restreindre la visualisation et l'accès aux rendez-vous et vacances pour une ou des UFs pour un utilisateur donné (Page 5, Guide paramétrage restrictions de visibilité agenda (1).pdf)
* On n'affiche que les rendez-vous associés à des séjours dont l'UFH est accessible par l'utilisateur (Page 6, Guide paramétrage restrictions de visibilité agenda (1).pdf)
* On n'affiche que les vacances dont l'UFH paramétrée est accessible